In [ ]:
# Step 1: Install required packages (run once)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install wandb -q  # optional
!pip install datasets transformers sentencepiece protobuf -q

# Step 2: Import libraries
import torch
from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
import pandas as pd
import os

# Step 3: Upload your CSV (run this cell and upload product_sft_data.csv)
from google.colab import files
uploaded = files.upload()  # In Kaggle: drag & drop to file panel

# Load your CSV
df = pd.read_csv("query-response-pairs-SFT-training-data.csv")  # Make sure filename matches exactly

# Optional: Inspect
print(f"Loaded {len(df)} examples")
print(df.head())

# Convert to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Step 4: Load model + tokenizer (4-bit for low VRAM)
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-7B-Instruct",
    max_seq_length=max_seq_length,
    dtype=None,           # Auto (bfloat16 if supported)
    load_in_4bit=True,    # ~6GB VRAM on T4/P100
    token=None,           # Add your HF token if private
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

# Step 5: Apply Qwen2.5 chat template to your query-response pairs
def format_chat(example):
    messages = [
        {"role": "system", "content": "You are an expert Product review assistant. Always be helpful, honest, and base your answers on real customer reviews."},
        {"role": "user",   "content": example["query"]},
        {"role": "assistant", "content": example["response"]}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

# Apply formatting
dataset = dataset.map(format_chat, remove_columns=dataset.column_names)

# Step 6: SFT Trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,   # Effective batch 16
        warmup_steps=10,
        max_steps=500,                   # Adjust based on your dataset size (e.g., 500–2000)
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="/content/qwen2.5-7b-amazon-assistant",
        report_to="none",                # Set to "wandb" after wandb.login()
        save_strategy="steps",
        save_steps=200,
        save_total_limit=3,
    ),
)

# Step 7: Start training
print("Starting fine-tuning on your Amazon product dataset...")
trainer_stats = trainer.train()

# Step 8: Save LoRA + Merged model
output_dir = "/content/Qwen2.5-7B-AmazonShoppingAssistant"

model.save_pretrained(output_dir + "-LoRA")
tokenizer.save_pretrained(output_dir + "-LoRA")

# Merge & save full 16-bit model (recommended)
model.save_pretrained_merged(
    output_dir,
    tokenizer,
    save_method="merged_16bit",
)
print(f"Full merged model saved to {output_dir}")

# Optional: Save GGUF for Ollama / llama.cpp
model.save_pretrained_gguf(
    output_dir + "-GGUF",
    tokenizer,
    quantization_method="q4_k_m"
)
print("GGUF version also saved!")

# Step 9: Quick inference test
FastLanguageModel.for_inference(model)

test_query = "Which is the best budget 43-inch 4K TV under ₹25,000 based on real customer reviews?"
messages = [
    {"role": "system", "content": "You are an expert Amazon India shopping assistant."},
    {"role": "user", "content": test_query}
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.9,
    do_sample=True
)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-wmwzx6z0/unsloth_1130764e50064cfea5dd9b9f1f00374a
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-wmwzx6z0/unsloth_1130764e50064cfea5dd9b9f1f00374a
  Resolved https://github.com/unslothai/unsloth.git to commit 6789c279d578278aca4af22f4ca31fc42829c9a4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.6/289.6 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 120.2 MB/s eta 0:00:00

Saving cc.csv to cc (1).csv
Loaded 100 examples
                                               query  \
0  Which is the best Lightning cable under ₹400 f...   
1  Suggest a durable Type-C cable under ₹200 that...   
2  I need a cable for both Micro USB and Type-C. ...   
3  Which cable is the toughest Micro USB one righ...   
4  I want a Lightning cable that supports fast ch...   

                                            response  
0  For daily use with iPhone 13 I always suggest ...  
1  Take the Ambrane Unbreakable 60W 1.5m Braided ...  
2  Go for the boAt Deuce USB 300 2-in-1 cable (Ma...  
3  The boAt Rugged V3 1.5 meter is currently the ...  
4  Portronics Konnect L 1.2M (grey or white) is p...  
==((====))==  Unsloth 2025.11.6: Fast Qwen2 patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None.

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [03:39<00:00, 54.93s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [04:19<00:00, 64.81s/it]


Unsloth: Merge process complete. Saved to `/content/Qwen2.5-7B-AmazonShoppingAssistant-GGUF`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: All required system packages already installed!
Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Cloning llama.cpp repository
Unsloth: Install GGUF and other packages


In [ ]:
from huggingface_hub import login, HfApi
from unsloth import FastLanguageModel

# Use your NEW token here
login("")

In [10]:
!pip install huggingface_hub hf-transfer -q

In [11]:
from huggingface_hub import login, HfApi
import os

In [12]:
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [ ]:
repo_id = "your-username/Qwen2.5-7B-ProductReviewAnalyzer-SFT-FP16"

In [14]:
api = HfApi()

In [ ]:
# FIXED UPLOAD CODE (compatible with latest huggingface_hub)
!pip install huggingface_hub hf-transfer -q --upgrade

from huggingface_hub import login, HfApi, create_repo
import os

# Login with your NEW token
login("")  # ← Replace with your actual token

# Enable fast upload
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

repo_id = "your-name/Qwen2.5-7B-ProductReviewAnalyzer-SFT-FP16"

api = HfApi()

# Create repo if needed
create_repo(repo_id=repo_id, repo_type="model", private=False, exist_ok=True)

print("Repo ready. Starting upload of full 16-bit model (~14 GB)...")
print("This will take 30–50 minutes. Do NOT stop the runtime!")

# SIMPLIFIED UPLOAD (no multi_commits)
api.upload_folder(
    folder_path="/content/Qwen2.5-7B-AmazonShoppingAssistant",
    repo_id=repo_id,
    commit_message="Qwen2.5-7B fine-tuned on real Amazon India product reviews | Full FP16 merged | Product Review Analyzer",
)

print("Upload complete!")
print(f"Model live at: https://huggingface.co/{repo_id}")

Repo ready. Starting upload of full 16-bit model (~14 GB)...
This will take 30–50 minutes. Do NOT stop the runtime!


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...gAssistant/tokenizer.json:   0%|          | 27.6kB / 11.4MB            

  ...0004-of-00004.safetensors:   2%|2         | 25.2MB / 1.09GB            

  ...0002-of-00004.safetensors:   0%|          |  611kB / 4.93GB            

  ...0003-of-00004.safetensors:   0%|          |  608kB / 4.33GB            

  ...0001-of-00004.safetensors:   0%|          | 16.8MB / 4.88GB            

Upload complete!
Model live at: https://huggingface.co/MuhammadHaaris/Qwen2.5-7B-ProductReviewAnalyzer-SFT-FP16
